# 07 · V4 với loss che vùng bằng chứng cú pháp

Bật GPU và Internet. Notebook lấy code từ nhánh `version4` trên GitHub. Điền `A1_CHECKPOINT` bằng đường dẫn Kaggle Input tới `a1_seed{SEED}.pt` hoặc ZIP chứa checkpoint cùng seed. Train nhánh RGCN từ đầu trên HC3 và dùng lại A1.

RGCN chạy một lượt cho mỗi mẫu trong bước train. Head tính điểm bằng chứng có dấu ở từng node, rồi lấy trung bình để tạo logit human. Loss bổ sung buộc model phân loại sau khi che một vùng tại bước tổng hợp. Mặc định `ERASURE_MODE='worst'`, ngân sách tối đa 20% node, trọng số loss 0,5 và 32 tâm ứng viên trải đều theo thứ tự node. Vùng là lân cận vô hướng, bán kính tối đa bằng số layer RGCN; nếu vượt ngân sách thì dùng bán kính nhỏ hơn. Model chọn vùng có BCE cao nhất trong các ứng viên bằng nhãn train. Với văn bản quá ngắn để che một node, loss bổ sung bằng 0.

Đổi `ERASURE_MODE='none'` để thử head-only hoặc `'random'` để chọn ngẫu nhiên trong cùng tập vùng ứng viên. Mỗi lần chạy dùng một chế độ. Các giá trị trên là cấu hình khởi đầu, cần chọn bằng validation HC3. Biểu diễn sau RGCN có thể đã chứa thông tin từ vùng bị che; hiệu quả chống chỉnh sửa văn bản thật cần thực nghiệm.

Cấu hình train dùng `struct=0`, `aug=none`, EMA và chọn checkpoint theo validation sạch + validation swap như V4. Deterministic mặc định tắt. Test dùng toàn bộ bằng chứng, nhân xác suất human với A1 và ngưỡng 0,5. Ô cuối hiển thị EMA riêng và EMA × A1 trên HC3 sạch, GPT-3.5 0% và HC3 Replace 25%, rồi xuất ZIP kết quả cùng cấu hình, lịch sử train và commit code.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4_evidence_erasure')
REPO = WORK / 'PRDetect-Research'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
BRANCH = 'version4'
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
                   check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', REPO_URL, BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'PRDetect {BRANCH}: {commit}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 2026
V3_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1
ERASURE_MODE = 'worst'  # 'none' = head-only; 'random' = random region; 'worst' = hardest region
ERASURE_FRACTION = 0.20
ERASURE_WEIGHT = 0.50
ERASURE_CANDIDATES = 32
A1_CHECKPOINT = None  # e.g. '/kaggle/input/your-dataset/a1_seed2026.pt', or a ZIP
RUN_NAME = f'evidence_{ERASURE_MODE}_rho{ERASURE_FRACTION:g}_lambda{ERASURE_WEIGHT:g}_k{ERASURE_CANDIDATES}'
CHECKPOINT_DIR = WORK / RUN_NAME / 'checkpoints'
FEATURE_CACHE = WORK / 'feature_cache'


In [ ]:
if not A1_CHECKPOINT:
    raise ValueError('Set A1_CHECKPOINT to your same-seed A1 file in /kaggle/input.')
a1_path = Path(A1_CHECKPOINT)
if not a1_path.is_file():
    raise FileNotFoundError(a1_path)
if a1_path.suffix.lower() == '.zip':
    name = f'a1_seed{SEED}.pt'
    with ZipFile(a1_path) as reference:
        matches = [n for n in reference.namelist() if Path(n).name == name]
        if len(matches) != 1:
            raise ValueError(f'Expected one {name} in {a1_path}, found {matches}')
        a1_path = WORK / name
        a1_path.write_bytes(reference.read(matches[0]))

cmd = [sys.executable, 'train_v4.py', '--seed', str(SEED),
       '--output-dir', str(CHECKPOINT_DIR), '--feature-cache', str(FEATURE_CACHE),
       '--v3-epochs', str(V3_EPOCHS), '--v3-struct', '0', '--v3-aug', 'none',
       '--v3-readout', 'evidence', '--a1-checkpoint', str(a1_path),
       '--erasure-mode', ERASURE_MODE, '--erasure-fraction', str(ERASURE_FRACTION),
       '--erasure-weight', str(ERASURE_WEIGHT), '--erasure-candidates', str(ERASURE_CANDIDATES),
       '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
       '--val-batch-size', str(VAL_BATCH_SIZE)]
subprocess.run(cmd, cwd=REPO, check=True)
run_dir = CHECKPOINT_DIR / f'seed{SEED}'
training = json.loads((run_dir / 'v4_train.json').read_text(encoding='utf-8'))
assert training['args']['v3_readout'] == 'evidence'
assert training['args']['erasure_mode'] == ERASURE_MODE
assert training['args']['a1_checkpoint']
archive = run_dir / f'v4_evidence_{ERASURE_MODE}_hc3_seed{SEED}.zip'
display(FileLink(str(archive)))

history = json.loads((run_dir / f'v3_baseline_dual_seed{SEED}.train.json').read_text(encoding='utf-8'))
print(f"Train time: {history['train_seconds'] / 60:.1f} min")
print(f"Best EMA epoch: {history['best_ema_epoch']}; best RAW epoch: {history['best_raw_epoch']}")
for row in history['history']:
    print(row['epoch'], row['evidence'])


In [ ]:
OUT = WORK / RUN_NAME / f'evidence_{ERASURE_MODE}_three_tests_seed{SEED}'
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'three',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', str(SEED), '--batch-size', str(EVAL_BATCH_SIZE),
                '--device', 'cuda', '--output-dir', str(OUT)], cwd=REPO, check=True)
import pandas as pd
results = pd.read_csv(OUT / 'results_by_seed.csv')
expected = {('table2', 'Original'), ('table3', 'Original'), ('table4', 'Replace 25%')}
if len(results) != 3 or set(zip(results['table'], results['condition'])) != expected:
    raise ValueError('Kết quả không khớp ba tập test của notebook 07.')
print('EMA x A1')
display(results)
solo = []
for row in results.to_dict('records'):
    report = json.loads((OUT / f'seed{SEED}' / f"{Path(row['source']).stem}_v3.json").read_text(encoding='utf-8'))
    assert report['readout'] == 'evidence'
    assert report['erasure_mode'] == ERASURE_MODE
    solo.append(dict(table=row['table'], condition=row['condition'], seed=SEED,
                     **{key: report[key] for key in ('accuracy', 'f1', 'auc')}))
solo = pd.DataFrame(solo)
print('EMA')
display(solo)
solo.to_csv(OUT / 'ema_results_by_seed.csv', index=False)
for name in ('v4_train.json', f'v3_baseline_dual_seed{SEED}.train.json'):
    shutil.copyfile(run_dir / name, OUT / name)
(OUT / 'code_commit.txt').write_text(commit + '\n', encoding='utf-8')
archive = shutil.make_archive(str(OUT), 'zip', OUT.parent, OUT.name)
display(FileLink(archive))